## Step 5 — spatially constrained multivariate clustering
**# of cells in notebook:** 1

**Purpose:** Classify buildings within each selected block into spatially contiguous groups based on calculated building neighborhood variables. Multiple clustering solutions are created so that later steps can evaluate alternative numbers of clusters.

**Input:**

- `building_context_features_new_dist.gpkg` from Step 4 for each selected block
- calculated building neighborhood variables describing:
  - building area
  - building coverage within the tessellation cell
  - polygon-edge nearest-neighbor distances
  - neighboring building-area characteristics
  - nearby building counts and building-area totals

**Output:**

Within each block folder:

- `spatial_multivariate_clusters_new_dist.gdb`
- clustered building feature classes for k = 2, 3, 4, and 5:
  - `building_scmc_newdist_k2_min10`
  - `building_scmc_newdist_k3_min10`
  - `building_scmc_newdist_k4_min10`
  - `building_scmc_newdist_k5_min10`
- ArcGIS processing/message text files for each clustering run

At the base block directory:

- `spatial_multivariate_clusters_new_dist_min10_summary.csv`

**Main logic:**

**Cell 1 — Run spatially constrained clustering**

1. Finds the Step 4 building-context dataset for each selected block and verifies that the required analysis fields are present.
2. Excludes buildings with null values in any analysis variable.
3. Runs ArcGIS Spatially Constrained Multivariate Clustering for k = 2, 3, 4, and 5 using a trimmed Delaunay triangulation spatial constraint.
4. Requires a minimum of 10 buildings in each cluster and skips a value of k when the block does not contain enough complete records.
5. Saves the clustered building layers, ArcGIS processing messages, and a summary of successful, skipped, or failed runs.


In [ ]:
import arcpy
import os
import glob
import csv
import traceback
import shutil

# ------------------------------------------------------------
# User inputs
# ------------------------------------------------------------

base_folder = r"E:\_kigali\_analysis\heterogeneous_largePop_blocks"

input_gpkg_name = "building_context_features_new_dist.gpkg"
preferred_input_layer = "building_context_features_new_dist"

output_gdb_name = "spatial_multivariate_clusters_new_dist.gdb"

# Run k = 2, 3, 4, 5 for every block
k_values = [2, 3, 4, 5]

# Spatially Constrained Multivariate Clustering settings
spatial_constraints = "TRIMMED_DELAUNAY_TRIANGULATION"
number_of_permutations = 100

# Cluster size constraint:
# every cluster must contain at least 10 features.
# There is no maximum cluster size.
size_constraints = "NUM_FEATURES"
min_features_per_cluster = 10
max_features_per_cluster = None

overwrite_outputs = True

# Do not add outputs to the active ArcGIS Pro map
add_outputs_to_map = False

analysis_fields = [
    "log_area_m2",
    "bldg_coverage_ratio",
    "dist_nn5",
    "dist_nn10",
    "dist_nn40",
    "mean_log_area_nn5",
    "mean_log_area_nn10",
    "mean_log_area_nn40",
    "count_bldgs_30m",
    "count_bldgs_60m",
    "count_bldgs_100m",
    "sum_area_30m",
    "sum_area_60m",
    "sum_area_100m"
]

# ------------------------------------------------------------
# ArcPy environment
# ------------------------------------------------------------

arcpy.env.overwriteOutput = overwrite_outputs
arcpy.env.parallelProcessingFactor = "50%"  # helpful because probabilities can be slow

# Prevent outputs from being added to the current ArcGIS Pro map.
# This environment is available in ArcGIS Pro sessions; wrapped defensively for stand-alone use.
try:
    arcpy.env.addOutputsToMap = add_outputs_to_map
except Exception:
    pass

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def find_block_folders(base_folder):
    """
    Finds block folders like:
      E:\\...\\large_pop_blocks\\_397
    """
    folders = []

    for folder in sorted(glob.glob(os.path.join(base_folder, "_*"))):
        if os.path.isdir(folder):
            folders.append(folder)

    return folders


def resolve_gpkg_feature_class(gpkg_path, preferred_layer):
    """
    Tries to resolve a GeoPackage feature class path.

    Usually this works:
      building_context_features_new_dist.gpkg\\building_context_features_new_dist

    But this function also searches the GeoPackage if the direct path
    is not recognized.
    """

    direct_path = os.path.join(gpkg_path, preferred_layer)

    if arcpy.Exists(direct_path):
        return direct_path

    # Try listing feature classes inside the GeoPackage
    old_workspace = arcpy.env.workspace

    try:
        arcpy.env.workspace = gpkg_path
        fcs = arcpy.ListFeatureClasses()

        if fcs:
            # Exact match first
            for fc in fcs:
                if fc.lower() == preferred_layer.lower():
                    return os.path.join(gpkg_path, fc)

            # Then allow partial match
            for fc in fcs:
                if preferred_layer.lower() in fc.lower():
                    return os.path.join(gpkg_path, fc)

            # Final fallback: if there is only one feature class, use it
            if len(fcs) == 1:
                return os.path.join(gpkg_path, fcs[0])

    finally:
        arcpy.env.workspace = old_workspace

    return None


def recreate_output_gdb(block_folder, output_gdb_name):
    """
    Deletes and recreates the output geodatabase inside the block folder.

    This intentionally overwrites the existing spatial_multivariate_clusters_new_dist.gdb.
    """
    out_gdb = os.path.join(block_folder, output_gdb_name)

    if arcpy.Exists(out_gdb):
        arcpy.management.Delete(out_gdb)
    elif os.path.exists(out_gdb):
        # Fallback in case ArcPy does not recognize the GDB for some reason.
        shutil.rmtree(out_gdb)

    arcpy.management.CreateFileGDB(
        out_folder_path=block_folder,
        out_name=output_gdb_name
    )

    return out_gdb


def check_required_fields(feature_class, required_fields):
    """
    Checks whether all required analysis fields exist.
    """
    existing = {f.name for f in arcpy.ListFields(feature_class)}
    missing = [f for f in required_fields if f not in existing]
    return missing


def make_not_null_where_clause(feature_class, fields):
    """
    Builds a where clause requiring all analysis fields to be non-null.

    This helps avoid failures if some buildings have missing values.
    """
    clauses = []

    for field in fields:
        delim = arcpy.AddFieldDelimiters(feature_class, field)
        clauses.append(f"{delim} IS NOT NULL")

    return " AND ".join(clauses)


def count_features(feature_class_or_layer):
    """
    Returns feature count.
    """
    return int(arcpy.management.GetCount(feature_class_or_layer)[0])


def write_messages_to_txt(txt_path, messages):
    """
    Writes ArcPy tool messages to a text file.
    """
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(messages)


# ------------------------------------------------------------
# Main workflow
# ------------------------------------------------------------

block_folders = find_block_folders(base_folder)

print("=" * 80)
print("Spatially Constrained Multivariate Clustering - New Distance Variables")
print("=" * 80)
print(f"Base folder: {base_folder}")
print(f"Block folders found: {len(block_folders):,}")
print(f"Input GeoPackage name: {input_gpkg_name}")
print(f"Input layer: {preferred_input_layer}")
print(f"Output GDB name: {output_gdb_name}")
print(f"Spatial constraints: {spatial_constraints}")
print(f"Permutations: {number_of_permutations}")
print(f"k values: {k_values}")
print(f"Cluster size constraint: minimum {min_features_per_cluster} features per cluster")
print("Maximum features per cluster: none")
print(f"Add outputs to map: {add_outputs_to_map}")

summary_rows = []

for block_folder in block_folders:

    block_id = os.path.basename(block_folder)

    print("\n" + "=" * 80)
    print(f"Processing block: {block_id}")
    print("=" * 80)

    gpkg_path = os.path.join(block_folder, input_gpkg_name)

    if not os.path.exists(gpkg_path):
        msg = f"Input GeoPackage not found: {gpkg_path}"
        print(f"  SKIPPING: {msg}")

        summary_rows.append({
            "block_folder": block_id,
            "block_folder_path": block_folder,
            "k": None,
            "status": "SKIPPED",
            "message": msg,
            "input_features": "",
            "output_features": "",
            "input_count": None,
            "selected_count": None,
            "output_count": None,
            "messages_txt": ""
        })

        continue

    input_fc = resolve_gpkg_feature_class(
        gpkg_path=gpkg_path,
        preferred_layer=preferred_input_layer
    )

    if input_fc is None:
        msg = f"Could not resolve feature class inside GeoPackage: {gpkg_path}"
        print(f"  SKIPPING: {msg}")

        summary_rows.append({
            "block_folder": block_id,
            "block_folder_path": block_folder,
            "k": None,
            "status": "SKIPPED",
            "message": msg,
            "input_features": "",
            "output_features": "",
            "input_count": None,
            "selected_count": None,
            "output_count": None,
            "messages_txt": ""
        })

        continue

    print("  Input features:")
    print(f"  {input_fc}")

    missing_fields = check_required_fields(input_fc, analysis_fields)

    if missing_fields:
        msg = f"Missing required fields: {missing_fields}"
        print(f"  SKIPPING: {msg}")

        summary_rows.append({
            "block_folder": block_id,
            "block_folder_path": block_folder,
            "k": None,
            "status": "SKIPPED",
            "message": msg,
            "input_features": input_fc,
            "output_features": "",
            "input_count": None,
            "selected_count": None,
            "output_count": None,
            "messages_txt": ""
        })

        continue

    input_count = count_features(input_fc)
    print(f"  Input feature count: {input_count:,}")

    # Make a feature layer excluding records with null analysis fields
    # so the clustering tool receives complete numeric records.
    where_clause = make_not_null_where_clause(input_fc, analysis_fields)

    layer_name = f"bldg_context_newdist_{block_id}_lyr"
    if arcpy.Exists(layer_name):
        arcpy.management.Delete(layer_name)

    arcpy.management.MakeFeatureLayer(
        in_features=input_fc,
        out_layer=layer_name,
        where_clause=where_clause
    )

    selected_count = count_features(layer_name)
    print(f"  Non-null analysis records: {selected_count:,}")

    if selected_count < 3:
        msg = "Fewer than 3 records with complete analysis fields."
        print(f"  SKIPPING: {msg}")

        summary_rows.append({
            "block_folder": block_id,
            "block_folder_path": block_folder,
            "k": None,
            "status": "SKIPPED",
            "message": msg,
            "input_features": input_fc,
            "output_features": "",
            "input_count": input_count,
            "selected_count": selected_count,
            "output_count": None,
            "messages_txt": ""
        })

        if arcpy.Exists(layer_name):
            arcpy.management.Delete(layer_name)

        continue

    # Recreate output GDB after input validation.
    # This overwrites the existing spatial_multivariate_clusters_new_dist.gdb.
    try:
        out_gdb = recreate_output_gdb(block_folder, output_gdb_name)
        print("  Output GDB recreated:")
        print(f"  {out_gdb}")
    except Exception as e:
        msg = f"Could not recreate output GDB: {e}"
        print(f"  SKIPPING: {msg}")

        summary_rows.append({
            "block_folder": block_id,
            "block_folder_path": block_folder,
            "k": None,
            "status": "ERROR",
            "message": msg,
            "input_features": input_fc,
            "output_features": "",
            "input_count": input_count,
            "selected_count": selected_count,
            "output_count": None,
            "messages_txt": ""
        })

        if arcpy.Exists(layer_name):
            arcpy.management.Delete(layer_name)

        continue

    # --------------------------------------------------------
    # Run k = 2, 3, 4, 5
    # --------------------------------------------------------

    for k in k_values:

        # With a minimum of 10 features per cluster, the absolute minimum
        # complete-record count is k * 10.
        required_min_records = k * min_features_per_cluster

        if selected_count < required_min_records:
            msg = (
                f"Not enough complete records for k={k} with minimum "
                f"{min_features_per_cluster} features per cluster. "
                f"Need at least {required_min_records}, found {selected_count}."
            )
            print(f"  SKIPPING k={k}: {msg}")

            summary_rows.append({
                "block_folder": block_id,
                "block_folder_path": block_folder,
                "k": k,
                "status": "SKIPPED",
                "message": msg,
                "input_features": input_fc,
                "output_features": "",
                "input_count": input_count,
                "selected_count": selected_count,
                "output_count": None,
                "messages_txt": ""
            })

            continue

        out_name = f"building_scmc_newdist_k{k}_min10"
        out_fc = os.path.join(out_gdb, out_name)

        msg_txt = os.path.join(
            block_folder,
            f"spatial_multivariate_clusters_new_dist_k{k}_min10_messages.txt"
        )

        if arcpy.Exists(out_fc):
            if overwrite_outputs:
                arcpy.management.Delete(out_fc)
            else:
                msg = f"Output exists and overwrite_outputs=False: {out_fc}"
                print(f"  SKIPPING k={k}: {msg}")

                summary_rows.append({
                    "block_folder": block_id,
                    "block_folder_path": block_folder,
                    "k": k,
                    "status": "SKIPPED",
                    "message": msg,
                    "input_features": input_fc,
                    "output_features": out_fc,
                    "input_count": input_count,
                    "selected_count": selected_count,
                    "output_count": None,
                    "messages_txt": msg_txt
                })

                continue

        print("\n  " + "-" * 72)
        print(f"  Running Spatially Constrained Multivariate Clustering: k={k}")
        print(f"  Minimum features per cluster: {min_features_per_cluster}")
        print(f"  Output: {out_fc}")

        try:
            arcpy.stats.SpatiallyConstrainedMultivariateClustering(
                in_features=layer_name,
                output_features=out_fc,
                analysis_fields=analysis_fields,
                size_constraints=size_constraints,
                constraint_field=None,
                min_constraint=min_features_per_cluster,
                max_constraint=max_features_per_cluster,
                number_of_clusters=k,
                spatial_constraints=spatial_constraints,
                weights_matrix_file=None,
                number_of_permutations=number_of_permutations,
                output_table=None
            )

            messages = arcpy.GetMessages()
            write_messages_to_txt(msg_txt, messages)

            out_count = count_features(out_fc)

            print(f"  SUCCESS k={k}")
            print(f"  Output feature count: {out_count:,}")
            print("  Messages written to:")
            print(f"  {msg_txt}")

            summary_rows.append({
                "block_folder": block_id,
                "block_folder_path": block_folder,
                "k": k,
                "status": "SUCCESS",
                "message": "Completed successfully.",
                "input_features": input_fc,
                "output_features": out_fc,
                "input_count": input_count,
                "selected_count": selected_count,
                "output_count": out_count,
                "messages_txt": msg_txt
            })

        except Exception as e:
            messages = arcpy.GetMessages()
            error_text = traceback.format_exc()

            combined_error = (
                f"Python error:\n{error_text}\n\n"
                f"ArcPy messages:\n{messages}"
            )

            write_messages_to_txt(msg_txt, combined_error)

            print(f"  ERROR k={k}")
            print(f"  {e}")
            print("  Error/messages written to:")
            print(f"  {msg_txt}")

            summary_rows.append({
                "block_folder": block_id,
                "block_folder_path": block_folder,
                "k": k,
                "status": "ERROR",
                "message": str(e),
                "input_features": input_fc,
                "output_features": out_fc,
                "input_count": input_count,
                "selected_count": selected_count,
                "output_count": None,
                "messages_txt": msg_txt
            })

    # Clean up layer
    if arcpy.Exists(layer_name):
        arcpy.management.Delete(layer_name)

# ------------------------------------------------------------
# Write summary CSV
# ------------------------------------------------------------

summary_csv = os.path.join(
    base_folder,
    "spatial_multivariate_clusters_new_dist_min10_summary.csv"
)

fieldnames = [
    "block_folder",
    "block_folder_path",
    "k",
    "status",
    "message",
    "input_features",
    "output_features",
    "input_count",
    "selected_count",
    "output_count",
    "messages_txt"
]

with open(summary_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()

    for row in summary_rows:
        writer.writerow({field: row.get(field, "") for field in fieldnames})

print("\n" + "=" * 80)
print("Done.")
print("=" * 80)
print("Summary written to:")
print(summary_csv)
